# ProtMamba Ghost Detection: Spin Test + Context Tax

**Hypothesis**: ProtMamba's geometric stability may be a **"Ghost"** —
a geometrically self-consistent manifold that is not grounded in biological reality.

## Two Tests

### Test 1: Self-Procrustes "Spin" Test
For each sequence length (synthetic) / length bin (UniRef), compute
`Procrustes(Clean, Perturbed)` for each perturbation type.

- **Low residual** = the embedding has a fixed orientation (grounded).
- **High residual** = the model creates a perfect geometric shape that
  **spins freely** — it has no biological anchor.

### Test 2: Frozen Head "Context Tax" Test
Train a simple linear classifier (Frozen Head) on ProtMamba embeddings for
a task that only requires **local information** (~50 aa signal).

The **Padding Strategy** (mirrors AlphaGenome Ghost Detection):
- 100 aa case: [50 aa Signal] + [50 aa Random Padding]
- 200 aa case: [50 aa Signal] + [150 aa Random Padding]
- 400 aa case: [50 aa Signal] + [350 aa Random Padding]
- 800 aa case: [50 aa Signal] + [750 aa Random Padding]

Task: Distinguish real **E. coli** proteins from real **Human** proteins.
A 50 aa region contains enough species-specific motifs for easy classification.

## Data Sources

- **Test 1 (Spin)**: Uses cached `.npy` embeddings from Scaling/UniRef experiments
- **Test 2 (Frozen Head)**: Requires **live ProtMamba inference** (GPU needed)

## Prerequisites

- Run after `ProtMamba_Scaling_Experiment.ipynb` (synthetic)
  and/or `ProtMamba_UniRef_Experiment.ipynb` (real) for Test 1.
- Test 2 requires GPU and ProtMamba model weights.

---

In [ ]:
import sys, os
print('Runtime ready')

In [ ]:
# Install Dependencies

print('Installing dependencies...')
!pip install -q torch transformers matplotlib seaborn pandas scipy scikit-learn einops biopython
!pip install -q causal-conv1d>=1.1.0
!pip install -q mamba-ssm

import os
if not os.path.exists('ProtMamba-ssm'):
    !git clone https://github.com/Bitbol-Lab/ProtMamba-ssm.git
!pip install -q -e ProtMamba-ssm/

WEIGHTS_DIR = 'protmamba_weights'
if not os.path.exists(WEIGHTS_DIR):
    os.makedirs(WEIGHTS_DIR, exist_ok=True)
    !wget -q -O protmamba_weights.zip https://github.com/Bitbol-Lab/ProtMamba-ssm/releases/download/v1.0/ProtMamba_model-weights.zip
    !unzip -q -o protmamba_weights.zip -d {WEIGHTS_DIR}
    !rm protmamba_weights.zip

import numpy as np
import torch
import matplotlib
import matplotlib.pyplot as plt
import pandas as pd
from scipy.linalg import orthogonal_procrustes
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
import seaborn as sns

matplotlib.rcParams.update({'font.size': 11})
print(f'torch {torch.__version__} | CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
print('Done!')

In [ ]:
# Configuration

# --- Paths to cached ProtMamba embeddings (for Test 1) ---
SCALING_CACHE = './results/protmamba_scaling_experiment_bins/cache'
UNIREF_CACHE = './results/protmamba_uniref_experiment_bins/cache'

OUTPUT_BASE = './results/protmamba_ghost_detection/'
RESULTS_DIR = OUTPUT_BASE + 'results'
CACHE_DIR = OUTPUT_BASE + 'cache'

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)

# Test 1: Synthetic sequence lengths and UniRef length bins
SEQ_LENGTHS = [100, 200, 300, 400, 500, 600, 700, 800]
LENGTH_BINS = [
    (50, 150), (151, 250), (251, 350), (351, 450),
    (451, 550), (551, 650), (651, 800),
]
PERTURBATIONS = ['aa_sub_1pct', 'aa_sub_2pct', 'aa_sub_5pct', 'aa_sub_10pct', 'reverse']

# Test 2: Frozen Head config
N_CLASSIFICATION_SEQS = 200      # per class (200 ecoli + 200 human = 400 total)
SIGNAL_LENGTH = 50               # 50 aa signal region
TARGET_LENGTHS = [100, 200, 400, 800]  # total sequence lengths to test
N_CV_FOLDS = 5

print(f'Output: {RESULTS_DIR}')
print(f'Scaling cache: {SCALING_CACHE}')
print(f'UniRef cache: {UNIREF_CACHE}')
print(f'Frozen Head: {SIGNAL_LENGTH} aa signal, padded to {TARGET_LENGTHS}')

In [ ]:
# Discover Cached Embeddings (for Test 1)

def discover_synthetic():
    found = {}
    for L in SEQ_LENGTHS:
        clean_path = f'{SCALING_CACHE}/protmamba_L{L}_clean.npy'
        if os.path.exists(clean_path):
            data = {'clean': clean_path}
            X = np.load(clean_path)
            data['n_sequences'] = X.shape[0]
            data['embed_dim'] = X.shape[1]
            for pert in PERTURBATIONS:
                pert_path = f'{SCALING_CACHE}/protmamba_L{L}_{pert}.npy'
                if os.path.exists(pert_path):
                    data[pert] = pert_path
            found[L] = data
    return found

def discover_uniref():
    found = {}
    for lo, hi in LENGTH_BINS:
        clean_path = f'{UNIREF_CACHE}/protmamba_uniref_bin{lo}_{hi}_clean.npy'
        if os.path.exists(clean_path):
            data = {'clean': clean_path}
            X = np.load(clean_path)
            data['n_sequences'] = X.shape[0]
            data['embed_dim'] = X.shape[1]
            for pert in PERTURBATIONS:
                pert_path = f'{UNIREF_CACHE}/protmamba_uniref_bin{lo}_{hi}_{pert}.npy'
                if os.path.exists(pert_path):
                    data[pert] = pert_path
            found[(lo, hi)] = data
    return found

synth_data = discover_synthetic()
uniref_data = discover_uniref()

print('=== SYNTHETIC (Scaling Experiment) ===')
if synth_data:
    for L, data in sorted(synth_data.items()):
        n_perts = len([k for k in data if k not in ('clean', 'n_sequences', 'embed_dim')])
        print(f'  L={L}: {data["n_sequences"]} seqs x {data["embed_dim"]}d ({n_perts} perturbations)')
else:
    print('  No synthetic data found. Run ProtMamba_Scaling_Experiment first.')

print(f'\n=== REAL (UniRef Experiment) ===')
if uniref_data:
    for (lo, hi), data in sorted(uniref_data.items()):
        n_perts = len([k for k in data if k not in ('clean', 'n_sequences', 'embed_dim')])
        print(f'  [{lo}-{hi}): {data["n_sequences"]} seqs x {data["embed_dim"]}d ({n_perts} perturbations)')
else:
    print('  No UniRef data found. Run ProtMamba_UniRef_Experiment first.')

if not synth_data and not uniref_data:
    print('\nWARNING: No cached embeddings for Test 1. Test 2 can still run independently.')

In [ ]:
# TEST 1 — Self-Procrustes Spin Test

def compute_procrustes_metrics(X_clean, X_pert, n_samples=2000):
    X_clean = np.nan_to_num(X_clean, nan=0.0, posinf=1e6, neginf=-1e6)
    X_pert = np.nan_to_num(X_pert, nan=0.0, posinf=1e6, neginf=-1e6)
    n = min(X_clean.shape[0], X_pert.shape[0])
    X_clean, X_pert = X_clean[:n], X_pert[:n]
    if n > n_samples:
        rng = np.random.default_rng(320)
        idx = rng.choice(n, n_samples, replace=False)
        X_clean, X_pert = X_clean[idx], X_pert[idx]
    X_c = X_clean - X_clean.mean(axis=0)
    X_p = X_pert - X_pert.mean(axis=0)
    raw_error = np.linalg.norm(X_c - X_p, 'fro') / np.sqrt(len(X_c))
    scale_c = np.linalg.norm(X_c, 'fro')
    scale_p = np.linalg.norm(X_p, 'fro')
    if scale_c < 1e-12 or scale_p < 1e-12:
        return {'raw_error': raw_error, 'aligned_error': raw_error,
                'ratio': 1.0, 'reduction_pct': 0, 'optimal_scale': 1.0, 'residual': 1.0}
    X_cn = X_c / scale_c
    X_pn = X_p / scale_p
    R, _ = orthogonal_procrustes(X_pn, X_cn)
    X_aligned = X_p @ R
    s = np.trace(X_c.T @ X_aligned) / np.trace(X_aligned.T @ X_aligned)
    X_aligned_scaled = X_aligned * s
    aligned_error = np.linalg.norm(X_c - X_aligned_scaled, 'fro') / np.sqrt(len(X_c))
    ratio = aligned_error / raw_error if raw_error > 1e-12 else 1.0
    residual = np.linalg.norm(X_cn - X_pn @ R, 'fro') ** 2
    return {
        'raw_error': raw_error, 'aligned_error': aligned_error,
        'ratio': ratio, 'reduction_pct': (1.0 - ratio) * 100,
        'optimal_scale': s, 'residual': residual,
    }

# ─── Run Procrustes on SYNTHETIC ───
print('=' * 70)
print('TEST 1: PROCRUSTES SPIN TEST — SYNTHETIC')
print('=' * 70)

proc_rows_synth = []
for L in sorted(synth_data.keys()):
    data = synth_data[L]
    X_clean = np.load(data['clean'])
    print(f'\n--- L={L} (n={X_clean.shape[0]}) ---')
    for pert in PERTURBATIONS:
        if pert not in data: continue
        X_pert = np.load(data[pert])
        result = compute_procrustes_metrics(X_clean, X_pert)
        proc_rows_synth.append({
            'source': 'synthetic', 'seq_length': L,
            'n_sequences': data['n_sequences'],
            'perturbation': pert, **result,
        })
        print(f'  {pert:>15}: ratio={result["ratio"]:.3f}  '
              f'residual={result["residual"]:.4f}  '
              f'scale={result["optimal_scale"]:.4f}')

# ─── Run Procrustes on UNIREF ───
print(f'\n{"=" * 70}')
print('TEST 1: PROCRUSTES SPIN TEST — UNIREF (REAL)')
print('=' * 70)

proc_rows_real = []
for (lo, hi) in sorted(uniref_data.keys()):
    data = uniref_data[(lo, hi)]
    X_clean = np.load(data['clean'])
    label = f'{lo}-{hi}'
    print(f'\n--- [{label}) (n={data["n_sequences"]}) ---')
    for pert in PERTURBATIONS:
        if pert not in data: continue
        X_pert = np.load(data[pert])
        result = compute_procrustes_metrics(X_clean, X_pert)
        proc_rows_real.append({
            'source': 'uniref', 'length_bin': label,
            'bin_lo': lo, 'bin_hi': hi,
            'n_sequences': data['n_sequences'],
            'perturbation': pert, **result,
        })
        print(f'  {pert:>15}: ratio={result["ratio"]:.3f}  '
              f'residual={result["residual"]:.4f}  '
              f'scale={result["optimal_scale"]:.4f}')

df_synth = pd.DataFrame(proc_rows_synth)
df_real = pd.DataFrame(proc_rows_real)
df_synth.to_csv(f'{RESULTS_DIR}/ghost_procrustes_synthetic.csv', index=False)
df_real.to_csv(f'{RESULTS_DIR}/ghost_procrustes_uniref.csv', index=False)
print(f'\nSaved {len(df_synth)} synthetic + {len(df_real)} real rows')

In [ ]:
# TEST 1 — Visualization

PERT_COLORS = {
    'reverse': '#DC2626',
    'aa_sub_1pct': '#93C5FD', 'aa_sub_2pct': '#60A5FA',
    'aa_sub_5pct': '#3B82F6', 'aa_sub_10pct': '#1D4ED8',
}

fig, axes = plt.subplots(2, 2, figsize=(16, 11))

# Top-Left: Synthetic Procrustes Residual
ax = axes[0, 0]
if len(df_synth) > 0:
    for pert in PERTURBATIONS:
        sub = df_synth[df_synth['perturbation'] == pert].sort_values('seq_length')
        if sub.empty: continue
        ax.plot(sub['seq_length'], sub['residual'], 'o-', linewidth=2, markersize=7,
                color=PERT_COLORS.get(pert, '#6B7280'), label=pert.replace('_', ' '))
ax.set_xlabel('Sequence Length (aa)'); ax.set_ylabel('Procrustes Residual')
ax.set_title('A. Synthetic: Spin Test', fontweight='bold')
ax.legend(fontsize=8); ax.grid(True, alpha=0.15)

# Top-Right: Real Procrustes Residual
ax = axes[0, 1]
if len(df_real) > 0:
    for pert in PERTURBATIONS:
        sub = df_real[df_real['perturbation'] == pert].sort_values('bin_lo')
        if sub.empty: continue
        centers = (sub['bin_lo'] + sub['bin_hi']) / 2
        ax.plot(centers, sub['residual'], 'o-', linewidth=2, markersize=7,
                color=PERT_COLORS.get(pert, '#6B7280'), label=pert.replace('_', ' '))
ax.set_xlabel('Length Bin Center (aa)'); ax.set_ylabel('Procrustes Residual')
ax.set_title('B. UniRef (Real): Spin Test', fontweight='bold')
ax.legend(fontsize=8); ax.grid(True, alpha=0.15)

# Bottom-Left: Synthetic Ratio
ax = axes[1, 0]
if len(df_synth) > 0:
    for pert in PERTURBATIONS:
        sub = df_synth[df_synth['perturbation'] == pert].sort_values('seq_length')
        if sub.empty: continue
        ax.plot(sub['seq_length'], sub['ratio'], 's-', linewidth=2, markersize=7,
                color=PERT_COLORS.get(pert, '#6B7280'), label=pert.replace('_', ' '))
ax.axhline(y=1.0, color='#94A3B8', linestyle=':', linewidth=1)
ax.set_xlabel('Sequence Length (aa)'); ax.set_ylabel('Procrustes Ratio')
ax.set_title('C. Synthetic: How Much Can Rotation Fix?', fontweight='bold')
ax.set_ylim(0, 1.15); ax.legend(fontsize=8); ax.grid(True, alpha=0.15)

# Bottom-Right: Real Ratio
ax = axes[1, 1]
if len(df_real) > 0:
    for pert in PERTURBATIONS:
        sub = df_real[df_real['perturbation'] == pert].sort_values('bin_lo')
        if sub.empty: continue
        centers = (sub['bin_lo'] + sub['bin_hi']) / 2
        ax.plot(centers, sub['ratio'], 's-', linewidth=2, markersize=7,
                color=PERT_COLORS.get(pert, '#6B7280'), label=pert.replace('_', ' '))
ax.axhline(y=1.0, color='#94A3B8', linestyle=':', linewidth=1)
ax.set_xlabel('Length Bin Center (aa)'); ax.set_ylabel('Procrustes Ratio')
ax.set_title('D. UniRef (Real): How Much Can Rotation Fix?', fontweight='bold')
ax.set_ylim(0, 1.15); ax.legend(fontsize=8); ax.grid(True, alpha=0.15)

fig.suptitle('ProtMamba Ghost Detection: Self-Procrustes Spin Test',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/protmamba_ghost_spin.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

# Summary
print('\n' + '=' * 70)
print('SPIN TEST SUMMARY (Reverse perturbation)')
print('=' * 70)
for label, df in [('Synthetic', df_synth), ('UniRef', df_real)]:
    rev = df[df['perturbation'] == 'reverse'].copy()
    if rev.empty: continue
    print(f'\n  {label}:')
    for _, row in rev.iterrows():
        tag = row.get('seq_length', row.get('length_bin', '?'))
        print(f'    {tag}: residual={row["residual"]:.4f}  ratio={row["ratio"]:.3f}')
    if len(rev) >= 2:
        first, last = rev.iloc[0], rev.iloc[-1]
        if last['residual'] > first['residual'] * 1.1:
            print(f'    → DETACHING: residual grows {last["residual"]/first["residual"]:.2f}x')
        elif last['residual'] < first['residual'] * 0.9:
            print(f'    → GROUNDING: residual shrinks')
        else:
            print(f'    → NEUTRAL: residual roughly constant')

In [ ]:
# TEST 2 Setup — ProtMamba Model Loader
#
# Load ProtMamba for live inference (required for Frozen Head test).
# This requires GPU.

import gc, glob, time

def find_checkpoint_path(weights_dir='protmamba_weights'):
    for pattern in [
        f'{weights_dir}/**/checkpoint*',
        f'{weights_dir}/**/model*.pt',
        f'{weights_dir}/**/pytorch_model.bin',
        f'{weights_dir}/**/*.pt',
    ]:
        matches = glob.glob(pattern, recursive=True)
        if matches:
            path = matches[0]
            return os.path.dirname(path) if os.path.isfile(path) else path
    return weights_dir


def make_protmamba_embedding_fn(batch_size=8, signal_length=None):
    """Load ProtMamba and return embedding function."""
    import sys; sys.path.append(os.path.abspath('ProtMamba-ssm'))
    from ProtMamba_ssm.modules import MambaLMHeadModelwithPosids, load_model
    from ProtMamba_ssm.utils import AA_TO_ID

    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f'Loading ProtMamba on {device}...')

    ckpt_path = find_checkpoint_path()
    model = load_model(
        ckpt_path, model_class=MambaLMHeadModelwithPosids,
        device=device, dtype=torch.float16, checkpoint_mixer=False,
    )
    model.eval()

    n_layers = model.config.n_layer
    n_params = sum(p.numel() for p in model.parameters()) / 1e6
    print(f'  Params: {n_params:.1f}M | Layers: {n_layers}')

    aa_to_id = AA_TO_ID.copy()
    unk_id = aa_to_id.get('X', aa_to_id.get('<unk>', 0))

    @torch.no_grad()
    def embed(sequences):
        embeddings = []
        n_batches = (len(sequences) + batch_size - 1) // batch_size
        for i in range(0, len(sequences), batch_size):
            batch_seqs = sequences[i:i + batch_size]
            batch_idx = i // batch_size + 1
            if batch_idx % 20 == 0 or batch_idx == n_batches:
                print(f'    Batch {batch_idx}/{n_batches}', end='\r')
            max_len = max(len(s) for s in batch_seqs)
            token_ids = []
            for seq in batch_seqs:
                ids = [aa_to_id.get(aa, unk_id) for aa in seq]
                ids += [0] * (max_len - len(ids))
                token_ids.append(ids)
            tokens = torch.tensor(token_ids, dtype=torch.long, device=device)
            pos_ids = torch.arange(max_len, device=device).unsqueeze(0).expand(len(batch_seqs), -1)
            hidden_states = model(
                input_ids=tokens, save_layer=[n_layers], position_ids=pos_ids,
            )
            last_hidden = hidden_states[n_layers]
            batch_embeds = []
            for j, seq in enumerate(batch_seqs):
                if signal_length is not None and signal_length < len(seq):
                    # LOCAL POOLING: Only pool the signal region (first signal_length tokens)
                    pooled = last_hidden[j, :signal_length, :].mean(axis=0)
                else:
                    pooled = last_hidden[j, :len(seq), :].mean(axis=0)
                batch_embeds.append(pooled)
            embeddings.append(np.stack(batch_embeds))
        print()
        return np.concatenate(embeddings, axis=0)

    return embed

print('ProtMamba loader ready')

In [ ]:
# TEST 2 — Download Classification Protein Sequences
#
# Task: Distinguish real E. coli proteins from real Human proteins.
# Both are real biological sequences, but from different organisms.
# A ~50 aa region contains enough species-specific motifs for classification.
#
# Padding strategy (mirrors AlphaGenome):
#   Place the 50 aa signal at the START, pad the rest with random amino acids.

import urllib.request
import gzip
import re

AMINO_ACIDS = list('ACDEFGHIKLMNPQRSTVWY')

# UniProt reference proteome URLs (reviewed, Swiss-Prot)
ECOLI_URL = 'https://rest.uniprot.org/uniprotkb/stream?format=fasta&query=(organism_id:83333)+AND+(reviewed:true)&size=500'
HUMAN_URL = 'https://rest.uniprot.org/uniprotkb/stream?format=fasta&query=(organism_id:9606)+AND+(reviewed:true)&size=500'


def download_protein_sequences(url, n_sequences, signal_length, seed, cache_tag):
    """Download protein sequences from UniProt and extract signal-length regions."""
    cache_file = f'{CACHE_DIR}/{cache_tag}_{n_sequences}_{signal_length}.txt'
    if os.path.exists(cache_file):
        print(f'  Loading cached: {cache_file}')
        with open(cache_file) as f:
            return [line.strip() for line in f if line.strip()]

    print(f'  Downloading from UniProt...')
    req = urllib.request.Request(url, headers={'User-Agent': 'Python/alphagenome'})
    with urllib.request.urlopen(req) as response:
        raw = response.read().decode('ascii')

    # Parse FASTA
    proteins = []
    current_seq = []
    for line in raw.split('\n'):
        if line.startswith('>'):
            if current_seq:
                proteins.append(''.join(current_seq))
            current_seq = []
        elif line.strip():
            current_seq.append(line.strip().upper())
    if current_seq:
        proteins.append(''.join(current_seq))

    print(f'  Parsed {len(proteins)} proteins')

    # Filter: keep only proteins >= signal_length with standard AAs
    valid_aa = set(AMINO_ACIDS)
    valid_proteins = [p for p in proteins if len(p) >= signal_length
                      and all(c in valid_aa for c in p[:signal_length])]
    print(f'  Valid (>={signal_length} aa, standard AAs): {len(valid_proteins)}')

    # Sample signal regions: take the FIRST signal_length residues
    rng = np.random.default_rng(seed)
    if len(valid_proteins) > n_sequences:
        idx = rng.choice(len(valid_proteins), n_sequences, replace=False)
        valid_proteins = [valid_proteins[i] for i in idx]

    signals = [p[:signal_length] for p in valid_proteins[:n_sequences]]

    os.makedirs(CACHE_DIR, exist_ok=True)
    with open(cache_file, 'w') as f:
        f.write('\n'.join(signals))
    print(f'  Extracted {len(signals)} signal sequences ({signal_length} aa each)')
    return signals


def pad_protein_sequence(signal_seq, target_length, rng):
    """Pad a signal sequence with random amino acids to reach target_length."""
    pad_needed = target_length - len(signal_seq)
    if pad_needed <= 0:
        return signal_seq[:target_length]
    padding = ''.join(rng.choice(AMINO_ACIDS, size=pad_needed))
    return signal_seq + padding  # Signal at START, random padding after


print('Downloading classification sequences...')
print('\n--- E. coli (K-12) ---')
ecoli_signals = download_protein_sequences(
    ECOLI_URL, N_CLASSIFICATION_SEQS, SIGNAL_LENGTH, seed=320, cache_tag='ecoli_k12'
)
print(f'\n--- Human (H. sapiens) ---')
human_signals = download_protein_sequences(
    HUMAN_URL, N_CLASSIFICATION_SEQS, SIGNAL_LENGTH, seed=1991, cache_tag='human_hsapiens'
)

print(f'\nE. coli: {len(ecoli_signals)} sequences x {len(ecoli_signals[0])} aa')
print(f'Human:   {len(human_signals)} sequences x {len(human_signals[0])} aa')

# Labels: 0 = E. coli, 1 = Human
all_signals = ecoli_signals + human_signals
labels = np.array([0] * len(ecoli_signals) + [1] * len(human_signals))
print(f'Total: {len(all_signals)} sequences, classes: {np.bincount(labels)}')

In [ ]:
# TEST 2 — Embed Padded Sequences at Each Target Length
#
# For each target length:
#   1. Pad the 50 aa signal to the target length with random amino acids
#   2. Embed via ProtMamba
#   3. Cache the embeddings

# Two embedding modes:
# 1. LOCAL POOL: Pool only the signal region (fixes dilution argument)
# 2. GLOBAL POOL: Pool entire sequence (baseline for comparison)
embed_fn_local = make_protmamba_embedding_fn(batch_size=8, signal_length=SIGNAL_LENGTH)
embed_fn_global = make_protmamba_embedding_fn(batch_size=8, signal_length=None)

rng_pad = np.random.default_rng(320)
frozen_head_embeddings = {}  # target_length -> np.array

for target_len in TARGET_LENGTHS:
    cache_path = f'{CACHE_DIR}/frozen_head_L{target_len}.npy'

    if os.path.exists(cache_path):
        print(f'\nL={target_len}: Loading cached embeddings')
        frozen_head_embeddings[target_len] = np.load(cache_path)
        continue

    print(f'\n{"=" * 70}')
    print(f'L={target_len}: Padding {len(all_signals)} signals to {target_len} aa')
    print('=' * 70)

    padded = [pad_protein_sequence(s, target_len, rng_pad) for s in all_signals]
    print(f'  Padded sequence length: {len(padded[0])} aa')
    print(f'  Signal fraction: {SIGNAL_LENGTH/target_len*100:.1f}%')

    # --- Local pooling (only signal tokens) ---
    print(f'  Embedding via ProtMamba (LOCAL POOL: first {SIGNAL_LENGTH} tokens)...')
    embs = embed_fn_local(padded)
    embs = np.nan_to_num(embs, nan=0.0, posinf=1e6, neginf=-1e6)
    np.save(cache_path, embs)
    print(f'  Cached to {cache_path} | Shape: {embs.shape}')
    frozen_head_embeddings[target_len] = embs

    # --- Global pooling (entire sequence, for comparison) ---
    cache_global = f'{CACHE_DIR}/frozen_head_global_L{target_len}.npy'
    if not os.path.exists(cache_global):
        print(f'  Embedding via ProtMamba (GLOBAL POOL: all {target_len} tokens)...')
        embs_g = embed_fn_global(padded)
        embs_g = np.nan_to_num(embs_g, nan=0.0, posinf=1e6, neginf=-1e6)
        np.save(cache_global, embs_g)

print(f'\nEmbeddings ready for target lengths: {list(frozen_head_embeddings.keys())}')

In [ ]:
# TEST 2 — Train Frozen Linear Classifiers
#
# For each target length:
#   Train a LogisticRegression on the ProtMamba embeddings
#   Report cross-validated accuracy
#
# If Accuracy(800) < Accuracy(100): Context Tax is TOXIC.
# If Accuracy(800) ≈ Accuracy(100): Diminishing Returns.

print('=' * 70)
print('TEST 2: FROZEN HEAD — Context Tax Analysis')
print('=' * 70)

frozen_head_rows = []

for target_len in TARGET_LENGTHS:
    if target_len not in frozen_head_embeddings:
        print(f'\nL={target_len}: SKIPPED (no embeddings)')
        continue

    X = frozen_head_embeddings[target_len]
    y = labels[:X.shape[0]]

    print(f'\n--- L={target_len} aa ---')
    print(f'  Embedding shape: {X.shape}')
    print(f'  Signal fraction: {SIGNAL_LENGTH/target_len*100:.1f}%')

    clf = make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=1000, C=1.0, random_state=320)
    )
    cv = StratifiedKFold(n_splits=N_CV_FOLDS, shuffle=True, random_state=320)
    scores = cross_val_score(clf, X, y, cv=cv, scoring='accuracy')

    print(f'  Accuracy: {scores.mean():.4f} ± {scores.std():.4f}')
    print(f'  Per-fold: {[f"{s:.3f}" for s in scores]}')

    frozen_head_rows.append({
        'target_length': target_len,
        'signal_fraction': SIGNAL_LENGTH / target_len,
        'accuracy_mean': scores.mean(),
        'accuracy_std': scores.std(),
        'accuracy_min': scores.min(),
        'accuracy_max': scores.max(),
    })

df_frozen = pd.DataFrame(frozen_head_rows)
df_frozen.to_csv(f'{RESULTS_DIR}/ghost_frozen_head.csv', index=False)

# Verdict
print(f'\n{"=" * 70}')
print('FROZEN HEAD VERDICT')
print('=' * 70)
if len(df_frozen) >= 2:
    first = df_frozen.iloc[0]
    for _, row in df_frozen.iloc[1:].iterrows():
        delta = row['accuracy_mean'] - first['accuracy_mean']
        len_ratio = row['target_length'] / first['target_length']
        print(f'  L={int(first["target_length"])} -> L={int(row["target_length"])} '
              f'({len_ratio:.0f}x length): '
              f'Δaccuracy = {delta:+.4f}')
        if delta < -0.05:
            print(f'    -> TOXIC: Extra padding HURTS ({delta:.1%} drop). Ghost confirmed.')
        elif abs(delta) < 0.02:
            print(f'    -> DIMINISHING RETURNS: {len_ratio:.0f}x more tokens for ≈0 gain.')
        else:
            print(f'    -> HELPFUL: Extra context aids classification.')

print(f'\nSaved to {RESULTS_DIR}/ghost_frozen_head.csv')

In [ ]:
# TEST 2 — Visualization

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

# Panel A: Accuracy vs Sequence Length
ax = axes[0]
ax.errorbar(df_frozen['target_length'], df_frozen['accuracy_mean'],
            yerr=df_frozen['accuracy_std'],
            marker='o', linewidth=2.5, markersize=10, capsize=6,
            color='#2563EB', ecolor='#93C5FD', elinewidth=2)
ax.axhline(y=0.5, color='#DC2626', linestyle='--', linewidth=1, alpha=0.7, label='Random chance')

for _, row in df_frozen.iterrows():
    ax.annotate(f'{row["signal_fraction"]*100:.0f}% signal',
                xy=(row['target_length'], row['accuracy_mean']),
                xytext=(0, -20), textcoords='offset points',
                fontsize=8, color='#6B7280', ha='center')

ax.set_xlabel('Total Sequence Length (aa)', fontsize=11)
ax.set_ylabel('Frozen Head Accuracy (E. coli vs Human)', fontsize=11)
ax.set_title('A. Context Tax: Does More Padding Help?', fontweight='bold', fontsize=12)
ax.set_ylim(0.4, 1.05)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.15)

# Panel B: Signal Fraction vs Accuracy
ax = axes[1]
ax.scatter(df_frozen['signal_fraction'] * 100, df_frozen['accuracy_mean'],
           s=200, c='#2563EB', zorder=5, edgecolors='white', linewidths=2)
for _, row in df_frozen.iterrows():
    ax.annotate(f'L={int(row["target_length"])}',
                xy=(row['signal_fraction'] * 100, row['accuracy_mean']),
                xytext=(10, 5), textcoords='offset points',
                fontsize=10, fontweight='bold')

ax.set_xscale('log')
ax.set_xlabel('Signal Fraction (% of input that is real)', fontsize=11)
ax.set_ylabel('Accuracy', fontsize=11)
ax.set_title('B. Signal Dilution Curve', fontweight='bold', fontsize=12)
ax.axhline(y=0.5, color='#DC2626', linestyle='--', linewidth=1, alpha=0.7)
ax.set_ylim(0.4, 1.05)
ax.grid(True, alpha=0.15)

fig.suptitle('ProtMamba Ghost Detection: Frozen Head Context Tax',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/protmamba_ghost_frozen_head.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print(f'Saved {RESULTS_DIR}/protmamba_ghost_frozen_head.png')

In [ ]:
# Combined Figure — Spin Test + Context Tax

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# Panel A: Synthetic Spin (Reverse only)
ax = axes[0]
rev_synth = df_synth[df_synth['perturbation'] == 'reverse'].sort_values('seq_length') if len(df_synth) > 0 else pd.DataFrame()
if not rev_synth.empty:
    ax.plot(rev_synth['seq_length'], rev_synth['residual'],
            marker='D', linewidth=2.5, markersize=10, color='#DC2626')
    ax.fill_between(rev_synth['seq_length'], 0, rev_synth['residual'],
                    color='#DC2626', alpha=0.1)
ax.set_xlabel('Sequence Length (aa)')
ax.set_ylabel('Procrustes Residual')
ax.set_title('A. Spin Test (Synthetic)', fontweight='bold')
ax.grid(True, alpha=0.15)

# Panel B: Frozen Head Accuracy
ax = axes[1]
if len(df_frozen) > 0:
    ax.errorbar(df_frozen['target_length'], df_frozen['accuracy_mean'],
                yerr=df_frozen['accuracy_std'],
                marker='o', linewidth=2.5, markersize=10, capsize=6,
                color='#2563EB', ecolor='#93C5FD', elinewidth=2)
    ax.axhline(y=0.5, color='#DC2626', linestyle='--', linewidth=1, alpha=0.5)
ax.set_xlabel('Sequence Length (aa)')
ax.set_ylabel('Frozen Head Accuracy')
ax.set_title('B. Context Tax: Classification Utility', fontweight='bold')
ax.set_ylim(0.4, 1.05)
ax.grid(True, alpha=0.15)

# Panel C: Dual-axis (Residual + Accuracy)
ax1 = axes[2]
ax2 = ax1.twinx()
if not rev_synth.empty:
    ax1.plot(rev_synth['seq_length'], rev_synth['residual'],
             marker='D', linewidth=2.5, markersize=9, color='#DC2626',
             label='Procrustes Residual')
    ax1.set_ylabel('Procrustes Residual', color='#DC2626')
    ax1.tick_params(axis='y', labelcolor='#DC2626')
if len(df_frozen) > 0:
    ax2.plot(df_frozen['target_length'], df_frozen['accuracy_mean'],
             marker='o', linewidth=2.5, markersize=9, color='#2563EB',
             label='Frozen Head Accuracy')
    ax2.set_ylabel('Accuracy', color='#2563EB')
    ax2.tick_params(axis='y', labelcolor='#2563EB')
    ax2.set_ylim(0.4, 1.05)
    ax2.axhline(y=0.5, color='#2563EB', linestyle=':', linewidth=1, alpha=0.3)

ax1.set_xlabel('Sequence Length (aa)')
ax1.set_title('C. Combined: Spin vs. Utility', fontweight='bold')
ax1.grid(True, alpha=0.15)
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, fontsize=9, loc='center right')

fig.suptitle('ProtMamba Ghost Detection: Combined Evidence',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/protmamba_ghost_combined.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()